In [3]:
import os
print(rf"C:\Users\{os.getlogin()}\pulse\data\aggregated\transaction\country\india\state\\")

C:\Users\kartik\pulse\data\aggregated\transaction\country\india\state\\


In [1]:
import os
import json
import pandas as pd

# 1. Update this to your LOCAL 'aggregated/user' state folder
path = r"C:\Users\kartik\Downloads\pulse-master\pulse-master\data\aggregated\user\country\india\state"

user_data = []

if not os.path.exists(path):
    print("Path not found. Please verify the folder exists at:", path)
else:
    for state in os.listdir(path):
        p_state = os.path.join(path, state)
        if os.path.isdir(p_state):
            for year in os.listdir(p_state):
                p_year = os.path.join(p_state, year)
                if os.path.isdir(p_year):
                    for file in os.listdir(p_year):
                        p_file = os.path.join(p_year, file)
                        with open(p_file, 'r') as f:
                            d = json.load(f)
                            
                            # Core metrics
                            reg_users = d['data']['aggregated']['registeredUsers']
                            app_opens = d['data']['aggregated']['appOpens']
                            
                            # Device breakdown logic
                            # Note: Sometimes usersByDevice is None in the dataset
                            if d['data']['usersByDevice'] is not None:
                                for device in d['data']['usersByDevice']:
                                    user_data.append({
                                        "State": state,
                                        "Year": year,
                                        "Quarter": int(file.strip('.json')),
                                        "Registered_Users": reg_users,
                                        "App_Opens": app_opens,
                                        "Brand": device['brand'],
                                        "Count": device['count'],
                                        "Percentage": device['percentage']
                                    })

    # 2. Convert and Save
    df_user = pd.DataFrame(user_data)
    df_user.to_csv("phonepe_users.csv", index=False)
    print(f"Success! Saved {len(df_user)} rows to 'phonepe_users.csv'")

Success! Saved 6732 rows to 'phonepe_users.csv'


In [4]:
#MAP 

import os
import json
import pandas as pd

# CHANGE THIS PATH for each category (Map or Top)
path = r"C:\Users\kartik\Downloads\pulse-master\pulse-master\data\map\transaction\hover\country\india\state"

map_data = []

for state in os.listdir(path):
    p_state = os.path.join(path, state)
    if os.path.isdir(p_state):
        for year in os.listdir(p_state):
            p_year = os.path.join(p_state, year)
            for file in os.listdir(p_year):
                p_file = os.path.join(p_year, file)
                with open(p_file, 'r') as f:
                    d = json.load(f)
                    for i in d['data']['hoverDataList']:
                        map_data.append({
                            "State": state,
                            "Year": year,
                            "Quarter": int(file.strip('.json')),
                            "District": i['name'],
                            "Count": i['metric'][0]['count'],
                            "Amount": i['metric'][0]['amount']
                        })

df_map = pd.DataFrame(map_data)
df_map.to_csv("map_transactions.csv", index=False)
print("Saved map_transactions.csv")

Saved map_transactions.csv


In [5]:
# 'top' folder
path = r"C:\Users\kartik\Downloads\pulse-master\pulse-master\data\top\transaction\country\india\state"

top_data = []

for state in os.listdir(path):
    p_state = os.path.join(path, state)
    if os.path.isdir(p_state):
        for year in os.listdir(p_state):
            p_year = os.path.join(p_state, year)
            for file in os.listdir(p_year):
                p_file = os.path.join(p_year, file)
                with open(p_file, 'r') as f:
                    d = json.load(f)
                    for i in d['data']['pincodes']:
                        top_data.append({
                            "State": state,
                            "Year": year,
                            "Quarter": int(file.strip('.json')),
                            "Pincode": i['entityName'],
                            "Count": i['metric']['count'],
                            "Amount": i['metric']['amount']
                        })

df_top = pd.DataFrame(top_data)
df_top.to_csv("top_transactions.csv", index=False)
print("Saved top_transactions.csv")

Saved top_transactions.csv


In [13]:
import pandas as pd
from sqlalchemy import create_engine

# --- CONFIGURATION ---
# Replace 'root' with your MySQL username
# Replace 'your_password' with your MySQL password
# Replace 'phonepe_db' with your actual database name
user = "root"
password = "1234"
host = "localhost"
database = "phonepe_db"

# Create the connection engine
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{database}")

def upload_to_sql(file_name, table_name):
    try:
        # Read the CSV you created earlier
        df = pd.read_csv(file_name)
        
        # Push to SQL (if_exists='replace' creates the table for you)
        df.to_sql(table_name, engine, if_exists='replace', index=False)
        print(f"✅ Successfully uploaded {file_name} to table '{table_name}'")
    except Exception as e:
        print(f"❌ Error uploading {file_name}: {e}")

# --- EXECUTION ---
# List of all the CSVs you have created
files_to_upload = {
    "phonepe_transactions.csv": "aggregated_transaction",
    "phonepe_users.csv": "aggregated_user",
    "map_transactions.csv": "map_transaction",
    "top_transactions.csv": "top_transaction"
}

for csv_file, table_name in files_to_upload.items():
    upload_to_sql(csv_file, table_name)

print("\nAll data is now in your SQL database!")

❌ Error uploading phonepe_transactions.csv: (pymysql.err.OperationalError) (1045, "Access denied for user 'root'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)
❌ Error uploading phonepe_users.csv: (pymysql.err.OperationalError) (1045, "Access denied for user 'root'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)
❌ Error uploading map_transactions.csv: (pymysql.err.OperationalError) (1045, "Access denied for user 'root'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)
❌ Error uploading top_transactions.csv: (pymysql.err.OperationalError) (1045, "Access denied for user 'root'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

All data is now in your SQL database!


In [2]:
# --- CONFIGURATION ---
user = "root"          # Usually 'root'
password = "12345"  # Change this to your real MySQL password
host = "localhost"
database = "phonepe_db" # Ensure this database was created in MySQL first

In [1]:
import pymysql

try:
    conn = pymysql.connect(
        host='localhost',
        user='root',
        password='12345', # Change this
        database='phonepe_db'            # Change this
    )
    print("✅ Connection Successful!")
    conn.close()
except Exception as e:
    print(f"❌ Connection Failed: {e}")

✅ Connection Successful!


In [3]:
import pandas as pd
from sqlalchemy import create_engine

# --- CONFIGURATION ---
user = "root"
password = "12345" 
host = "localhost"
database = "phonepe_db"

# Create the engine using the verified credentials
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{database}")

def upload_to_sql(file_name, table_name):
    try:
        df = pd.read_csv(file_name)
        # 'replace' will create the table if it doesn't exist
        df.to_sql(table_name, engine, if_exists='replace', index=False)
        print(f"✅ Successfully uploaded {file_name} to table '{table_name}'")
    except Exception as e:
        print(f"❌ Error uploading {file_name}: {e}")

# --- EXECUTION ---
files_to_upload = {
    "phonepe_transactions.csv": "aggregated_transaction",
    "phonepe_users.csv": "aggregated_user",
    "map_transactions.csv": "map_transaction",
    "top_transactions.csv": "top_transaction"
}

for csv_file, table_name in files_to_upload.items():
    upload_to_sql(csv_file, table_name)

print("\n🚀 All data is now live in MySQL!")

✅ Successfully uploaded phonepe_transactions.csv to table 'aggregated_transaction'
✅ Successfully uploaded phonepe_users.csv to table 'aggregated_user'
✅ Successfully uploaded map_transactions.csv to table 'map_transaction'
✅ Successfully uploaded top_transactions.csv to table 'top_transaction'

🚀 All data is now live in MySQL!
